# Validating the fundamentals loader

This notebook checks `src/loaders/fundamentals.py` against real cached filings. It is the
companion to `tests/test_fundamentals.py`, and the two do different jobs. The pytest suite runs
against synthetic fixtures in under a second with no network, so it catches a regression the
moment it happens. This notebook runs against a fixed panel of real companies, which is what
establishes that a rule is correct in the first place and what would reveal a filing pattern
nobody anticipated.

Findings are written up in `notebooks/logs/fundamentals_construction.md`, Parts 14 to 17. This
file is the reproducible working; that one is the reasoning.

## How to run it

Run the cells in order. Section 1 must run first; every later section depends on it. Sections 3
and 4 are independent of each other.

Two conventions make it re-runnable:

**Everything comes from `src` through the `F.` prefix.** Nothing is redefined here. If a name is
not prefixed it is a fixture or a measurement helper local to this notebook, never a copy of
loader logic. `importlib.reload(F)` in section 1 picks up edits to the module without restarting
the kernel; a bare `from ... import name` would not, which is why the prefix is used throughout.

**Assertions use point in time queries with a fixed past `as_of` date.** A query asking what was
known on 2022-01-01 has a permanent answer, so it can be asserted. A query asking what is known
today will change as new filings arrive, so those are printed for inspection rather than asserted.
Mixing the two would produce a notebook that fails every quarter for no reason.

## 1. Setup

In [1]:
import collections
import datetime as dt
import importlib
import json
import os
import statistics
import time

import pandas as pd
import requests

# VS Code sets the kernel's working directory to the notebook's own folder, but src/ and every
# DATA_RAW / DATA_PROCESSED constant is written relative to the project root. Guarded so
# re-running does not walk up an extra level.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import src.loaders.fundamentals as F
from src.universe.point_in_time import build_universe

importlib.reload(F)   # picks up edits to the module without restarting the kernel
print(os.getcwd())


def plus_days(iso, n):
    """A date offset by n days, in the ISO string form used throughout the loader."""
    return (dt.date.fromisoformat(iso) + dt.timedelta(days=n)).isoformat()


def period_days(point):
    """Length of the period a fact covers, or None for an instant."""
    return F._period_days(point)


def concept_points(facts, concept, unit="USD"):
    """Every fact for a concept, across its aliases, flattened."""
    return [p for taxonomy, tag in F.TAG_ALIASES[concept]
            for p in facts["facts"].get(taxonomy, {}).get(tag, {}).get("units", {}).get(unit, [])]

/Users/hongxianli/Documents/data_science/capm-portfolio


## 2. The standing panel

Eighteen companies, each present for a recorded reason, covering the axes that have produced
failures: filing era, fiscal calendar shape, business model, share class structure, index status,
and specific known tagging pathologies.

The panel is selected to be difficult, so its pass rates measure whether an implementation handles
the shapes we know exist. They are not an estimate of how often each shape occurs, which is what a
full scale run against the real universe measures. Those stay two separate numbers.

In [2]:
PANEL = [
    ("AAPL", "fiscal calendar", "52/53 week year ending late September; also the real 27% FY2008 restatement"),
    ("COST", "fiscal calendar", "a 12-12-12-16 week retail year, so quarters run 83 days and the fourth 111"),
    ("KO",   "fiscal calendar", "conventional calendar year consumer goods filer, the control case"),
    ("GOOGL", "share structure", "dual class that does tag a combined undimensioned count; also the mid-history LongTermDebt switch and a 20:1 split"),
    ("META", "share structure", "dual class with no undimensioned share count in either taxonomy"),
    ("DASH", "filing era",      "first filed after ASC 606: no Revenues, no GrossProfit, convertible debt tags, no share count"),
    ("MWV",  "filing era",      "pre-2018 filer using the deprecated SalesRevenueNet tag; since departed"),
    ("CCU-200807", "taxonomy and currency", "universe misattribution to a foreign private issuer: files 20-F under ifrs-full in CLP, so every us-gaap alias returns None while coverage still records fetched: True"),
    ("JPM",  "business model",  "bank, calendar year end; FY2021 net income revised at the same duration"),
    ("FITB", "business model",  "bank with no revenue tag under any alias"),
    ("ACGL", "business model",  "insurer: revenue present, no gross profit path"),
    ("INVH", "business model",  "REIT reporting a predecessor and a successor entity either side of a 2017 reorganisation"),
    ("KKR",  "business model",  "alternative asset manager, no gross profit path"),
    ("CMG",  "business model",  "restaurant: operating costs presented without a gross profit subtotal"),
    ("CEG",  "business model",  "utility, no gross profit path"),
    ("KSU",  "index status",    "railroad, acquired and departed; no gross profit path"),
    ("MCK",  "known pathology", "reports Assets and StockholdersEquity but never an explicit Liabilities tag"),
    ("FAST", "known pathology", "the same balance sheet identity case, confirmed directly in Part 9"),
]

# Where ticker_history maps a panel ticker to more than one CIK, the ambiguity is resolved here
# explicitly rather than inside the lookup, so the reason sits next to the choice. GOOGL is the
# case the universe log documents: an administrative reorganisation moved the reporting entity
# from Google Inc (1288776) to Alphabet Inc (1652044) in 2015 with no trading discontinuity, and
# 1652044 is the entity every finding in the fundamentals log was established against.
CIK_OVERRIDES = {"GOOGL": 1652044}

print(len(PANEL), "panel members")

18 panel members


In [3]:
universe_spans, ticker_history = build_universe()


def ciks_for_ticker(ticker):
    """Reverse of ticker_on: which entities ever used a symbol.

    Returns every match on purpose. More than one means either the documented universe-module CIK
    misattribution or a genuinely recycled symbol, and silently taking the first would hide both.
    """
    matches = ticker_history.loc[ticker_history["ticker"] == ticker, "cik"].dropna().unique()
    return sorted(int(c) for c in matches)


rows = []
for ticker, axis, why in PANEL:
    ciks = ciks_for_ticker(ticker)
    if ticker in CIK_OVERRIDES:
        cik = CIK_OVERRIDES[ticker]
    elif len(ciks) == 1:
        cik = ciks[0]
    else:
        print(f"{ticker}: {len(ciks)} CIKs, needs an override before it joins the panel")
        continue

    # Cached to the same location the full build uses, so this fetch is not wasted: those CIKs
    # are already present when build_fundamentals runs. Re-running the cell costs nothing.
    facts = F.load_company_facts(cik)
    if facts is None:
        try:
            facts = F.fetch_company_facts(cik)
            F.save_company_facts(cik, facts)
        except requests.HTTPError as e:
            # No companyfacts at all, expected for a pre-2009 delisting rather than a failure.
            print(f"{ticker} (CIK {cik}): no companyfacts, HTTP {e.response.status_code}")
        time.sleep(F.REQUEST_PAUSE_SECONDS)

    rows.append({"ticker": ticker, "cik": cik, "axis": axis, "why": why,
                 "has_facts": facts is not None})

panel = pd.DataFrame(rows)
assert len(panel) == len(PANEL), "a panel member failed to resolve to a single CIK"
assert panel["has_facts"].all(), "a panel member returned no companyfacts"
print(f"{len(panel)} panel members resolved and cached")
panel

18 panel members resolved and cached


,ticker,cik,axis,why,has_facts
0,AAPL,320193,fiscal calendar,52/53 week year ending late September; also th...,True
1,COST,909832,fiscal calendar,"a 12-12-12-16 week retail year, so quarters ru...",True
2,KO,21344,fiscal calendar,conventional calendar year consumer goods file...,True
3,GOOGL,1652044,share structure,dual class that does tag a combined undimensio...,True
4,META,1326801,share structure,dual class with no undimensioned share count i...,True
5,DASH,1792789,filing era,"first filed after ASC 606: no Revenues, no Gro...",True
6,MWV,1159297,filing era,pre-2018 filer using the deprecated SalesReven...,True
7,CCU-200807,888746,taxonomy and currency,universe misattribution to a foreign private i...,True
8,JPM,19617,business model,"bank, calendar year end; FY2021 net income rev...",True
9,FITB,35527,business model,bank with no revenue tag under any alias,True


## 3. Period classification (Decision A)

An end date does not identify an income statement fact, because a quarter end carries both the
three month figure and the year to date figure. This section establishes what does identify one.

### 3.1 The duration distribution

In [4]:
def fact_rows(facts, concept, unit="USD"):
    """Every fact for a concept, with the fields that could plausibly identify a period type."""
    out = []
    for taxonomy, tag in F.TAG_ALIASES[concept]:
        for p in facts["facts"].get(taxonomy, {}).get(tag, {}).get("units", {}).get(unit, []):
            out.append({"tag": tag, "start": p.get("start"), "end": p["end"], "val": p["val"],
                        "filed": p["filed"], "form": p["form"], "fp": p.get("fp"),
                        "duration_days": period_days(p)})
    return pd.DataFrame(out)


CONCEPTS = ["net_income", "revenue", "operating_cash_flow", "total_assets"]

frames = []
for row in panel.itertuples():
    facts = F.load_company_facts(row.cik)
    for concept in CONCEPTS:
        df = fact_rows(facts, concept)
        if df.empty:
            continue
        df["ticker"], df["axis"], df["concept"] = row.ticker, row.axis, concept
        frames.append(df)

rows = pd.concat(frames, ignore_index=True)
print(rows.shape[0], "facts across", rows["ticker"].nunique(), "companies\n")

# Balance sheet facts have no "start" key and appear as one NaN bucket. Flow durations should
# form clusters with visible gaps between them; the width of each cluster is what any tolerance
# would have to accommodate, and the gaps are what make classification possible at all.
rows["duration_days"].value_counts(dropna=False).sort_index()

11043 facts across 17 companies



duration_days
30         3
58         2
83       504
86        16
87         9
88         9
89       870
90      1734
91      1151
92         9
97        27
111       49
118       11
149        2
167      128
177       13
178       11
179        9
180      653
181      306
182      112
183       20
188       18
241        2
251      122
252        4
268       13
269       12
270       11
271       14
272      705
273      231
274      116
279       18
333        2
363      319
364     1247
365      423
370       74
None    2064
Name: count, dtype: int64

Reading the result: `NaN` holds every balance sheet fact and nothing else, so an instant is
identified structurally rather than by tolerance. The flow durations cluster near 90, 180, 272,
and 364 days.

Note the off-by-one: `end - start` is one less than the inclusive day count, so a 52 week year
measures 363, a calendar year 364, a leap calendar year 365, and a 53 week year 370.

### 3.2 What the unusual durations are

In [5]:
# Bands drawn tight around the dense centres, so anything at an edge surfaces as unclustered and
# gets identified rather than absorbed. Provisional, for inspection only.
CORES = {"quarterly": (85, 95), "half_year": (175, 185),
         "nine_month": (265, 275), "annual": (360, 372)}


def core_of(d):
    if pd.isna(d):
        return "instant"
    for name, (low, high) in CORES.items():
        if low <= d <= high:
            return name
    return "unclustered"


rows["core"] = rows["duration_days"].map(core_of)
print(rows["core"].value_counts().to_string(), "\n")

edge = rows[rows["core"] == "unclustered"]
print(edge.groupby(["duration_days", "ticker", "concept"]).size().to_string())

core
quarterly      3798
instant        2064
annual         2063
half_year      1124
nine_month     1102
unclustered     892 

duration_days  ticker  concept            
30             INVH    net_income               3
58             INVH    net_income               2
83             COST    net_income             166
                       operating_cash_flow     26
                       revenue                312
97             AAPL    net_income              11
                       operating_cash_flow      5
                       revenue                 11
111            COST    net_income              18
                       revenue                 31
118            COST    net_income               4
                       revenue                  7
149            INVH    net_income               2
167            COST    net_income              34
                       operating_cash_flow     26
                       revenue                 68
188            AAPL    net_inc

Nearly all of the oddities come from three companies, and each has an explanation that generalises
beyond the individual filer.

**Costco** divides its fiscal year 12-12-12-16 weeks rather than into four equal quarters, so
three quarters run 83 days and the fourth runs 111, or 118 in a 53 week year. Its year to date
figures fall at 167, 251, and 363 days.

**Apple's 53 week years** open with a 14 week quarter, giving 97 days and year to date figures of
188 and 279 instead of the usual 180 and 272.

**Invitation Homes** reports a predecessor and a successor entity either side of a 2017
reorganisation: January 2017 alone at 30 days, a successor series from 2017-02-01 at 58, 149, 241,
and 333 days, and a combined calendar year at 364.

Together these establish that a quarter is 83 days at Costco, 89 to 91 at a calendar year filer,
97 at Apple, and 111 or 118 at Costco's fourth quarter. That is a 35 day spread for one period
type across two companies, which is why period type is measured as a fraction of the filer's own
year rather than in absolute days.

### 3.3 Why EDGAR's own `fp` and `form` fields cannot be used

In [6]:
# Both fields describe the filing a fact appeared in rather than the fact itself, so the same fact
# carries different values in each filing that re-reports it as a comparative. If a field gives
# several answers for one fact it cannot identify that fact.
costco = F.load_company_facts(int(panel.loc[panel["ticker"] == "COST", "cik"].iloc[0]))
target = ("2019-02-18", "2019-05-12")

variants = {(p["fp"], p["form"], p["fy"]) for p in concept_points(costco, "net_income")
            if (p.get("start"), p["end"]) == target}
print(f"Costco's fact for {target[0]} to {target[1]}, one start, one end, one value:")
for fp, form, fy in sorted(variants, key=str):
    print(f"   fp={fp}  form={form}  fy={fy}")

assert len({fp for fp, _, _ in variants}) > 1, "expected one fact to appear under several fp values"
print("\nthe same fact appears under multiple fp values, so fp cannot identify it")

Costco's fact for 2019-02-18 to 2019-05-12, one start, one end, one value:
   fp=FY  form=10-K  fy=2020
   fp=Q3  form=10-Q  fy=2019
   fp=Q3  form=10-Q  fy=2020
   fp=Q4  form=10-K  fy=2019

the same fact appears under multiple fp values, so fp cannot identify it


### 3.4 The start date tolerance

Two facts can share an end date and a period type for two different reasons, and they need
opposite treatment. One reporting period whose start was tagged inconsistently between filings
must be merged, so that the filing date rule can resolve it as the restatement it is. Two
genuinely different periods must be kept apart. The only property separating them is how far apart
the start dates are, and that threshold is measured rather than chosen.

In [7]:
def classify_panel(concept, unit="USD"):
    """Every distinct period for a concept across the panel, with its period type."""
    out = []
    for row in panel.itertuples():
        facts = F.load_company_facts(row.cik)
        points = concept_points(facts, concept, unit)
        if not points:
            continue
        year_days = F.annual_duration(facts)
        seen = set()
        for p in points:
            key = (p.get("start"), p["end"])
            if key in seen:      # the same period is re-reported in every later filing
                continue
            seen.add(key)
            out.append({"ticker": row.ticker, "concept": concept, "start": p.get("start"),
                        "end": p["end"], "days": period_days(p), "val": p["val"],
                        "label": F.period_type(p, year_days)})
    return pd.DataFrame(out)


multi = pd.concat([classify_panel(c) for c in
                   ["net_income", "revenue", "operating_cash_flow"]], ignore_index=True)


def start_gaps(frame, keys=("ticker", "concept", "end")):
    """Gaps between consecutive start dates within each group sharing an end date and a label.

    The concept must be part of the key: net income, revenue and operating cash flow for one
    quarter share a start and an end, so omitting it collects them into one group and reports
    spurious zero day gaps between facts that were never in competition.
    """
    out = []
    labelled = frame[frame["label"].notna() & (frame["label"] != "instant")]
    for group_key, g in labelled.groupby(list(keys) + ["label"]):
        g = g.sort_values("start")
        starts, vals = list(g["start"]), list(g["val"])
        for i in range(1, len(starts)):
            gap = (dt.date.fromisoformat(starts[i]) - dt.date.fromisoformat(starts[i - 1])).days
            row = dict(zip(list(keys) + ["label"], group_key))
            row.update({"gap_days": gap, "same_value": vals[i] == vals[i - 1]})
            out.append(row)
    return pd.DataFrame(out)


gaps = start_gaps(multi)
print(gaps.groupby(["gap_days", "same_value"]).size().to_string())

small = gaps[gaps["gap_days"] <= 8]
large = gaps[gaps["gap_days"] >= 31]
assert gaps[(gaps["gap_days"] > 8) & (gaps["gap_days"] < 31)].empty, \
    "the two groups have stopped being separable; the tolerance would need rethinking"
print(f"\nsmall gaps (1 to 8 days): {len(small)}, of which {int(small['same_value'].sum())} "
      f"carry identical values")
print(f"large gaps (31 days):     {len(large)}, of which {int(large['same_value'].sum())} "
      f"carry identical values")
print(f"\nnothing between 9 and 30 days, so START_TOLERANCE_DAYS "
      f"({F.START_TOLERANCE_DAYS}) is not a tuned parameter")

gap_days  same_value
1         False         1
          True          9
2         True          3
3         True          3
7         True          2
8         True          2
31        False         3

small gaps (1 to 8 days): 20, of which 19 carry identical values
large gaps (31 days):     3, of which 0 carry identical values

nothing between 9 and 30 days, so START_TOLERANCE_DAYS (15) is not a tuned parameter


### 3.5 The adopted rule, against the cases it was built for

In [8]:
# Point in time queries with fixed past as_of dates, so these answers are permanent and can be
# asserted. Each mirrors a filing pattern named in Part 14 of the log.

goog = F.load_company_facts(1652044)
aapl = F.load_company_facts(320193)
ko = F.load_company_facts(21344)
invh = F.load_company_facts(1687229)
cost = F.load_company_facts(int(panel.loc[panel["ticker"] == "COST", "cik"].iloc[0]))

checks = [
    # A quarter end carries both the three month and the year to date figure.
    ("Alphabet Q2 2021 quarterly", goog, "net_income", "2021-06-30", "2022-01-01", "quarterly", 18_525_000_000),
    ("Alphabet Q2 2021 half year", goog, "net_income", "2021-06-30", "2022-01-01", "half_year", 36_455_000_000),
    # A 12-12-12-16 week year: an 83 day quarter and a 167 day half year share an end date.
    ("Costco Q2 FY2020 quarterly", cost, "net_income", "2020-02-16", "2021-01-01", "quarterly", 931_000_000),
    ("Costco Q2 FY2020 half year", cost, "net_income", "2020-02-16", "2021-01-01", "half_year", 1_775_000_000),
    # A restatement whose start date shifted by a day in the revising filing.
    ("Coca-Cola Q2 2011, before the revision", ko, "net_income", "2011-07-01", "2011-09-01", "quarterly", 2_797_000_000),
    ("Coca-Cola Q2 2011, after the revision", ko, "net_income", "2011-07-01", "2013-01-01", "quarterly", 2_800_000_000),
    # Two reporting bases sharing an end date, 31 days apart in start.
    ("Invitation Homes FY2017", invh, "net_income", "2017-12-31", "2019-01-01", "annual", -105_337_000),
    # The point in time layer, unchanged by any of the above.
    ("Apple FY2008, before the 10-K/A", aapl, "net_income", "2008-09-27", "2009-12-01", "annual", 4_834_000_000),
    ("Apple FY2008, after the 10-K/A", aapl, "net_income", "2008-09-27", "2010-06-01", "annual", 6_119_000_000),
]

for label, facts, concept, period_end, as_of, period, expected in checks:
    got = F.concept_value_as_of(facts, concept, "USD", period_end, as_of, period)
    assert got is not None and got[0] == expected, f"{label}: expected {expected:,}, got {got}"
    print(f"  {label:42s} {got[0]:>18,}")

# Balance sheet concepts take no period and are unaffected.
assert F.concept_value_as_of(aapl, "total_assets", "USD", "2020-09-26", "2020-12-01")[0] == 323_888_000_000
assert F.total_liabilities_as_of(aapl, "2020-09-26", "2020-12-01") == 258_549_000_000
assert F.gross_profit_as_of(aapl, "2020-09-26", "2020-12-01", "annual") == 104_956_000_000
print("\ninstant concepts and the two derived helpers unchanged")

  Alphabet Q2 2021 quarterly                     18,525,000,000
  Alphabet Q2 2021 half year                     36,455,000,000
  Costco Q2 FY2020 quarterly                        931,000,000
  Costco Q2 FY2020 half year                      1,775,000,000
  Coca-Cola Q2 2011, before the revision          2,797,000,000
  Coca-Cola Q2 2011, after the revision           2,800,000,000
  Invitation Homes FY2017                          -105,337,000
  Apple FY2008, before the 10-K/A                 4,834,000,000
  Apple FY2008, after the 10-K/A                  6,119,000,000

instant concepts and the two derived helpers unchanged


### 3.6 Verification by accounting identity

The strongest available check, because it does not depend on how the classifier works: within a
fiscal year, consecutive year to date figures must differ by exactly the reported quarterly figure.
A classifier confusing the two could not satisfy this.

The exceptions are expected and bounded. A filer presenting statements in millions rounds each
figure independently, so three rounded numbers need not satisfy an exact identity, and a filer
revising one figure without the others leaves the latest vintage of each mutually inconsistent.
Both are sub-percent and neither is actionable in the loader.

In [9]:
LADDER = ["quarterly", "half_year", "nine_month", "annual"]


def identity_rows(ticker, cik, concept="net_income", unit="USD", as_of="2026-01-01"):
    """Check that year to date figures differ by the reported quarterly figure.

    A year to date fact shares its start with the quarter that opens the fiscal year, so grouping
    end dates by start date recovers one year's ladder without needing the filer's calendar.
    """
    facts = F.load_company_facts(cik)
    year_days = F.annual_duration(facts)
    if facts is None or year_days is None:
        return []
    points = concept_points(facts, concept, unit)

    by_start = {}
    for p in points:
        if p["filed"] <= as_of and F.period_type(p, year_days) in LADDER:
            by_start.setdefault(p["start"], set()).add(p["end"])

    out = []
    for start, ends in by_start.items():
        ladder_ends = sorted(ends)
        for i in range(1, min(len(ladder_ends), 4)):
            previous_end, this_end = ladder_ends[i - 1], ladder_ends[i]
            previous = F.concept_value_as_of(facts, concept, unit, previous_end, as_of, LADDER[i - 1])
            current = F.concept_value_as_of(facts, concept, unit, this_end, as_of, LADDER[i])
            quarter = F.concept_value_as_of(facts, concept, unit, this_end, as_of, "quarterly")
            if not (previous and current and quarter):
                continue
            implied = current[0] - previous[0]
            out.append({"ticker": ticker, "fy_start": start, "end": this_end,
                        "step": f"{LADDER[i]} minus {LADDER[i - 1]}",
                        "implied_quarter": implied, "reported_quarter": quarter[0],
                        "matches": implied == quarter[0],
                        "error_pct": 100 * (implied - quarter[0]) / quarter[0] if quarter[0] else None})
    return out


checks = []
for row in panel.itertuples():
    checks += identity_rows(row.ticker, row.cik)

ident = pd.DataFrame(checks)
rate = 100 * ident["matches"].mean()
worst = ident.loc[~ident["matches"], "error_pct"].abs().max()
print(f"{len(ident)} checks across {ident['ticker'].nunique()} companies")
print(f"balancing exactly: {int(ident['matches'].sum())} ({rate:.1f}%)")
print(f"largest discrepancy among the rest: {worst:.2f}%")

assert rate > 90, "the identity should hold for the great majority; a drop means misclassification"
assert worst < 5, "an exception larger than a few percent would be a whole-quarter error"
ident[~ident["matches"]].sort_values("error_pct", key=abs, ascending=False).head(10)

547 checks across 17 companies
balancing exactly: 523 (95.6%)
largest discrepancy among the rest: 1.03%


,ticker,fy_start,end,step,implied_quarter,reported_quarter,matches,error_pct
253,FITB,2020-01-01,2020-06-30,half_year minus quarterly,197000000,195000000,False,1.025641
411,CEG,2022-01-01,2022-09-30,nine_month minus half_year,-189000000,-188000000,False,0.531915
250,FITB,2018-01-01,2018-09-30,nine_month minus half_year,434000000,436000000,False,-0.458716
244,FITB,2015-01-01,2015-09-30,nine_month minus half_year,380000000,381000000,False,-0.262467
240,FITB,2013-01-01,2013-09-30,nine_month minus half_year,420000000,421000000,False,-0.237530
241,FITB,2014-01-01,2014-06-30,half_year minus quarterly,438000000,439000000,False,-0.227790
252,FITB,2019-01-01,2019-09-30,nine_month minus half_year,550000000,549000000,False,0.182149
262,FITB,2024-01-01,2024-09-30,nine_month minus half_year,572000000,573000000,False,-0.174520
254,FITB,2020-01-01,2020-09-30,nine_month minus half_year,580000000,581000000,False,-0.172117
261,FITB,2024-01-01,2024-06-30,half_year minus quarterly,602000000,601000000,False,0.166389


## 4. Shares outstanding (Decisions C and D)

Shares outstanding behaves unlike every other concept, for two reasons that compound: neither
source tag is dated to a period end a caller would hold, and for multi-class filers the count is
absent from the bulk endpoint entirely.

### 4.1 Why a period matched query cannot work

In [10]:
SHARES_TAGS = [("dei", "EntityCommonStockSharesOutstanding"),
               ("us-gaap", "CommonStockSharesOutstanding")]


def shares_points(facts, taxonomy, tag):
    return facts["facts"].get(taxonomy, {}).get(tag, {}).get("units", {}).get("shares", [])


def nearest_fact(points, target, as_of):
    """The visible fact whose end date is closest to `target`, and by how far."""
    visible = [p for p in points if p["filed"] <= as_of]
    if not visible:
        return None, None
    offset = lambda p: (dt.date.fromisoformat(p["end"]) - dt.date.fromisoformat(target)).days
    best = min(visible, key=lambda p: abs(offset(p)))
    return best["end"], offset(best)


rows = []
for row in panel.itertuples():
    facts = F.load_company_facts(row.cik)
    dei = shares_points(facts, *SHARES_TAGS[0])
    gaap = shares_points(facts, *SHARES_TAGS[1])

    year_days = F.annual_duration(facts)
    fy_ends = sorted({p["end"] for p in concept_points(facts, "net_income")
                      if F.period_type(p, year_days) == "annual"}) if year_days else []
    if not fy_ends:
        rows.append({"ticker": row.ticker, "fy_end": None, "dei_facts": len(dei),
                     "gaap_facts": len(gaap)})
        continue

    # A year after the period closed, so the annual report has certainly been filed. Querying as
    # of a fixed date shortly after the year end would measure filing lag, not the interface.
    fy_end = fy_ends[-1]
    as_of = plus_days(fy_end, 365)
    dei_end, dei_offset = nearest_fact(dei, fy_end, as_of)
    gaap_end, gaap_offset = nearest_fact(gaap, fy_end, as_of)

    rows.append({"ticker": row.ticker, "fy_end": fy_end, "dei_facts": len(dei),
                 "gaap_facts": len(gaap), "dei_offset": dei_offset, "gaap_offset": gaap_offset})

shares = pd.DataFrame(rows)
offsets = shares["dei_offset"].dropna()
print(f"cover page tag, distance from the fiscal year end: "
      f"{offsets.min():.0f} to {offsets.max():.0f} days, median {offsets.median():.0f}")
print(f"balance sheet tag, same measure: "
      f"{shares['gaap_offset'].dropna().unique()}")

assert (shares["gaap_offset"].dropna() == 0).all(), "the us-gaap tag should be dated to the period end"
assert (offsets.abs() > 0).all(), "the dei tag should never coincide with a period end"
print("\nThe cover page date is systematically later than the period end, not a noisy version of "
      "it,\nso no tolerance on period matching recovers it. Hence a separate function taking no "
      "period end.")
shares

cover page tag, distance from the fiscal year end: -55 to 54 days, median 30
balance sheet tag, same measure: [0.]

The cover page date is systematically later than the period end, not a noisy version of it,
so no tolerance on period matching recovers it. Hence a separate function taking no period end.


,ticker,fy_end,dei_facts,gaap_facts,dei_offset,gaap_offset
0,AAPL,2025-09-27,70,144,20.0,0.0
1,COST,2025-08-31,67,126,30.0,0.0
2,KO,2025-12-31,71,0,49.0,NaN
3,GOOGL,2025-12-31,0,88,NaN,0.0
4,META,2025-12-31,0,0,NaN,NaN
5,DASH,2025-12-31,0,0,NaN,NaN
6,MWV,2014-12-31,20,43,30.0,0.0
7,CCU-200807,NaN,10,0,NaN,NaN
8,JPM,2025-12-31,72,51,31.0,0.0
9,FITB,2025-12-31,68,138,32.0,0.0


### 4.2 Whether the two tags can be pooled

In [11]:
# Pooling and taking the freshest is only safe if both tags count the same thing. A filer whose
# cover page reported one share class while its balance sheet reported the combined total would be
# silently mixed, and market capitalisation would jump between rebalance dates according to which
# tag happened to be fresher.
pairs = []
for row in panel.itertuples():
    facts = F.load_company_facts(row.cik)
    dei = shares_points(facts, *SHARES_TAGS[0])
    gaap = shares_points(facts, *SHARES_TAGS[1])
    if not dei or not gaap:
        continue

    # The closest dated pair, so a genuine difference in scope is not confused with ordinary drift
    # in the share count between two distant dates.
    apart = lambda d, g: abs((dt.date.fromisoformat(d["end"])
                              - dt.date.fromisoformat(g["end"])).days)
    d, g = min(((d, g) for d in dei for g in gaap), key=lambda pair: apart(*pair))
    pairs.append({"ticker": row.ticker, "dei_end": d["end"], "gaap_end": g["end"],
                  "days_apart": apart(d, g), "dei_val": d["val"], "gaap_val": g["val"],
                  "ratio": round(d["val"] / g["val"], 4) if g["val"] else None})

pool = pd.DataFrame(pairs)
same_day = pool[pool["days_apart"] == 0]
print(f"{len(pool)} panel members carry both tags")
print(f"ratio range overall:            {pool['ratio'].min():.4f} to {pool['ratio'].max():.4f}")
print(f"ratio range where dated the same day: {same_day['ratio'].min():.4f} to "
      f"{same_day['ratio'].max():.4f}")

assert pool["ratio"].between(0.9, 1.1).all(), \
    "a ratio near 0.5 or 2 would mean one tag counts a single class and the other counts all"
print("\nBoth tags count the same thing, so they are pooled and the freshest wins. Deviations "
      "scale\nwith the gap between the dates, which is ordinary issuance and buyback drift.")
pool

11 panel members carry both tags
ratio range overall:            0.9991 to 1.0079
ratio range where dated the same day: 0.9991 to 1.0001

Both tags count the same thing, so they are pooled and the freshest wins. Deviations scale
with the gap between the dates, which is ordinary issuance and buyback drift.


,ticker,dei_end,gaap_end,days_apart,dei_val,gaap_val,ratio
0,AAPL,2009-06-27,2009-06-27,0,895816758,895735210,1.0001
1,COST,2014-05-28,2014-05-11,17,438295696,438582000,0.9993
2,MWV,2012-10-19,2012-09-30,19,174800536,174742488,1.0003
3,JPM,2010-01-31,2009-12-31,31,3973010673,3942000000,1.0079
4,FITB,2009-06-30,2009-06-30,0,795313448,795313448,1.0000
5,ACGL,2024-02-16,2023-12-31,47,374151215,373400000,1.0020
6,INVH,2018-03-22,2018-03-31,9,520364636,520364636,1.0000
7,KKR,2024-02-27,2024-03-31,33,885005588,885010967,1.0000
8,CEG,2022-03-31,2022-03-31,0,326698937,327000000,0.9991
9,KSU,2014-04-09,2014-03-31,9,110324940,110325979,1.0000


### 4.3 Frozen counts, and why staleness is bounded

In [12]:
# A count is only useful if it is roughly current. Some filers report an undimensioned count for
# part of their history and then switch to per-class tagging, so the endpoint keeps the old facts
# and nothing after. Without a bound the loader returns those as if current.
rows = []
for path in sorted(F.FUNDAMENTALS_RAW_DIR.glob("*.json")):
    facts = json.loads(path.read_text())
    # A filer with no us-gaap facts at all (an ifrs-full foreign private issuer, or an
    # unrelated entity reached through a misattributed CIK) was never a genuine candidate for
    # this concept, so it is excluded before anything else rather than counted as unresolved.
    if not facts["facts"].get("us-gaap"):
        continue
    last_filed = max((p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts), default=None)
    if last_filed is None:
        continue
    counts = [p for taxonomy, tag in SHARES_TAGS for p in shares_points(facts, taxonomy, tag)]
    if not counts:
        rows.append({"entity": facts.get("entityName", "?"), "state": "never reported",
                     "last_filing": last_filed, "last_count": None, "gap_days": None})
        continue
    best = max(counts, key=lambda p: (p["end"], p["filed"]))
    gap = (dt.date.fromisoformat(last_filed) - dt.date.fromisoformat(best["end"])).days
    rows.append({"entity": facts.get("entityName", "?"),
                 "state": "frozen" if gap > 400 else "current",
                 "last_filing": last_filed, "last_count": best["end"], "gap_days": gap})

freshness = pd.DataFrame(rows)
print(freshness["state"].value_counts().to_string())
print(f"\nof {len(freshness)} cached companies\n")

# Frozen counts are the dangerous case: a wrong number rather than a missing one.
print(freshness[freshness["state"] == "frozen"]
      .sort_values("gap_days", ascending=False).to_string(index=False))

for entity in ["Mastercard Incorporated", "Datadog, Inc."]:
    row = freshness[freshness["entity"] == entity]
    if not row.empty:
        assert row.iloc[0]["state"] == "frozen", f"{entity} was expected to be a frozen case"
print(f"\nMAX_SHARE_COUNT_AGE_DAYS = {F.MAX_SHARE_COUNT_AGE_DAYS} treats these as unavailable "
      "rather than returning them.")

state
current           788
frozen             36
never reported     29

of 853 cached companies

                            entity  state last_filing last_count  gap_days
               COMCAST CORPORATION frozen  2026-07-23 2009-12-31    6048.0
                    CME GROUP INC. frozen  2026-07-24 2010-02-17    6001.0
                         VISA INC. frozen  2026-04-29 2010-01-27    5936.0
                     Accenture plc frozen  2026-06-18 2010-03-19    5935.0
        United Parcel Service, Inc frozen  2026-05-07 2010-02-17    5923.0
          MOLSON COORS BEVERAGE CO frozen  2026-05-22 2010-04-29    5867.0
        Bio-Rad Laboratories, Inc. frozen  2026-04-30 2010-06-30    5783.0
           Mastercard Incorporated frozen  2026-07-30 2010-10-27    5755.0
                  Paramount Global frozen  2025-07-31 2010-04-30    5571.0
            BERKSHIRE HATHAWAY INC frozen  2026-05-04 2011-04-29    5484.0
                     Ford Motor Co frozen  2026-04-30 2011-04-28    5481.0
  

### 4.4 The period average fallback, and why quarterly rather than annual

Roughly 9 percent of filers report their count only per share class, so the bulk endpoint has
nothing. The weighted average share count is undimensioned for almost all of them, because
earnings per share requires it, but it answers a different question: the count averaged across a
reporting period rather than a balance at a date.

The objection is that its error should correlate with share repurchase, since an average over a
period exceeds the current count precisely when the count has been falling. That would penalise
repurchasers in the value factor, which is exactly the population value is meant to favour. The
measurement below tests both the bias and the mitigation.

In [13]:
WA_TAX, WA_TAG = F.WEIGHTED_AVERAGE_SHARES_TAG


def freshest_average(points, wanted, year_days, as_of):
    visible = [p for p in points if p["filed"] <= as_of and F.period_type(p, year_days) == wanted]
    return max(visible, key=lambda p: (p["end"], p["filed"]))["val"] if visible else None


paired = []
for path in sorted(F.FUNDAMENTALS_RAW_DIR.glob("*.json")):
    facts = json.loads(path.read_text())
    year_days = F.annual_duration(facts)
    if year_days is None:
        continue
    points = facts["facts"].get(WA_TAX, {}).get(WA_TAG, {}).get("units", {}).get("shares", [])
    if not points:
        continue
    for as_of in sorted({p["filed"] for p in points}):
        # The truth: a genuine point in time count, with the fallback switched off so that this
        # measurement cannot compare the approximation against itself.
        truth = F.shares_outstanding_as_of(facts, as_of, allow_weighted_average=False)
        if truth is None or truth[0] <= 0:
            continue
        a = freshest_average(points, "annual", year_days, as_of)
        q = freshest_average(points, "quarterly", year_days, as_of)
        if not (a and q):
            continue
        # A handful of filers tag the average in millions while the count is in units, so the two
        # are not on a common scale at all. Excluded rather than allowed to dominate the summary.
        if not (0.5 <= a / truth[0] <= 2.0 and 0.5 <= q / truth[0] <= 2.0):
            continue
        paired.append({"annual": 100 * (a - truth[0]) / truth[0],
                       "quarterly": 100 * (q - truth[0]) / truth[0]})

err = pd.DataFrame(paired)
summary = pd.DataFrame({
    "median": err.median().round(2),
    "median_abs": err.abs().median().round(2),
    "p05": err.quantile(0.05).round(2),
    "p95": err.quantile(0.95).round(2),
    "within_5pct": (err.abs() <= 5).mean().mul(100).round(0),
})
print(f"{len(err)} observations where both were available and a true count existed\n")
print(summary.to_string())
closer = (err["quarterly"].abs() < err["annual"].abs()).mean()
print(f"\nquarterly closer to the truth in {100 * closer:.0f}% of cases")

assert summary.loc["quarterly", "median_abs"] < summary.loc["annual", "median_abs"], \
    "the quarterly figure should be the more accurate of the two"
assert abs(summary.loc["quarterly", "median"]) < 0.2, \
    "a non-zero median would mean the repurchase skew has returned"
print("\nThe annual figure's positive median is the repurchase skew. The quarterly figure's "
      "median\nof zero is it disappearing: the lag falls from about ten months to about three.")

37714 observations where both were available and a true count existed

           median  median_abs   p05   p95  within_5pct
annual       0.37        1.73 -9.02  8.51         79.0
quarterly    0.03        0.43 -2.29  3.12         96.0

quarterly closer to the truth in 84% of cases

The annual figure's positive median is the repurchase skew. The quarterly figure's median
of zero is it disappearing: the lag falls from about ten months to about three.


### 4.5 The adopted rule, against the filers it was built for

In [14]:
# Which companies rely on the approximation, and what it gives them. Queried at each filer's own
# last filing date so that a departed name is not counted as a failure for having left the index.
rows = []
for path in sorted(F.FUNDAMENTALS_RAW_DIR.glob("*.json")):
    facts = json.loads(path.read_text())
    # Same exclusion as the frozen-counts cell above: a filer with no us-gaap facts at all
    # was never a genuine candidate for this concept.
    if not facts["facts"].get("us-gaap"):
        continue
    last_filed = max((p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts), default=None)
    if last_filed is None:
        continue
    strict = F.shares_outstanding_as_of(facts, last_filed, allow_weighted_average=False)
    if strict is not None:
        continue
    fallback = F.shares_outstanding_as_of(facts, last_filed)
    rows.append({"entity": facts.get("entityName", "?"),
                 "fallback": None if fallback is None else fallback[0],
                 "as_of": None if fallback is None else fallback[4],
                 "source": None if fallback is None else fallback[3]})

reliant = pd.DataFrame(rows)
print(f"{len(reliant)} of the cached companies have no genuine point in time count")
print(f"{reliant['fallback'].notna().sum()} of those are recovered by the fallback\n")
print(reliant.to_string(index=False))

assert len(reliant) / len(list(F.FUNDAMENTALS_RAW_DIR.glob("*.json"))) < 0.15, \
    "the share of filers relying on the approximation has grown unexpectedly"
assert reliant["fallback"].isna().sum() <= 10, \
    "unresolved beyond Sunoco (units, not shares) should only be filers with no usable " \
    "weighted-average tag, basic or diluted, at any age: dimensioned-only multi-class " \
    "filers (Berkshire, Ares, Visa, Constellation Brands, Ryan Specialty), " \
    "post-acquisition subsidiaries that stopped filing equity facts (Dell International, " \
    "Level 3 Parent), and a closed-end fund (First Trust Mortgage Income Fund). Tyson " \
    "Foods, resolved via the diluted-only tag, should no longer appear here."

66 of the cached companies have no genuine point in time count
56 of those are recovered by the fallback

                            entity     fallback      as_of                                          source
        Estee Lauder Companies Inc  362700000.0 2026-03-31   WeightedAverageNumberOfSharesOutstandingBasic
                 TYSON FOODS, INC.  354000000.0 2026-03-28 WeightedAverageNumberOfDilutedSharesOutstanding
          RALPH LAUREN CORPORATION   61100000.0 2025-12-27   WeightedAverageNumberOfSharesOutstandingBasic
                GRAHAM HOLDINGS CO    4265000.0 2026-06-30   WeightedAverageNumberOfSharesOutstandingBasic
      CABLEVISION SYSTEMS CORP /NY  271092000.0 2016-03-31   WeightedAverageNumberOfSharesOutstandingBasic
                     BROADCOM CORP  608000000.0 2015-09-30   WeightedAverageNumberOfSharesOutstandingBasic
        SIMON PROPERTY GROUP, INC.  324961423.0 2026-03-31   WeightedAverageNumberOfSharesOutstandingBasic
            BERKSHIRE HATHAWAY INC    

In [15]:
# Point in time assertions on the primary path, with fixed past as_of dates so the answers are
# permanent. Coca-Cola, Chipotle and McKesson report only the cover page tag, so a period matched
# query reached none of their counts before Decision C.
for entity_cik, as_of, expected_source in [
    (21344, "2021-06-01", "EntityCommonStockSharesOutstanding"),
    (1058090, "2021-06-01", "EntityCommonStockSharesOutstanding"),
    (1652044, "2021-06-01", "CommonStockSharesOutstanding"),
]:
    facts = F.load_company_facts(entity_cik)
    got = F.shares_outstanding_as_of(facts, as_of)
    assert got is not None, f"CIK {entity_cik} should resolve as of {as_of}"
    assert got[3] == expected_source, f"CIK {entity_cik} resolved via {got[3]}"
    print(f"  CIK {entity_cik:>8}  {got[0]:>18,}  end={got[4]}  via {got[3]}")

# Routing shares outstanding through the period matched interface is an error rather than a
# silent None, since that silence is the defect Decision C fixed.
try:
    F.concept_value_as_of(F.load_company_facts(21344), "shares_outstanding", "shares",
                          "2020-12-31", "2021-06-01")
    raise AssertionError("expected a ValueError")
except ValueError as e:
    print(f"\n  guard fires: {e}")

  CIK    21344       4,311,680,667  end=2021-04-23  via EntityCommonStockSharesOutstanding
  CIK  1058090          28,150,479  end=2021-04-26  via EntityCommonStockSharesOutstanding
  CIK  1652044         671,094,000  end=2021-03-31  via CommonStockSharesOutstanding

  guard fires: shares outstanding is not a period-matched concept; call shares_outstanding_as_of(facts, as_of_date) instead. The cover page tag is dated 20 to 54 days after the period end, so no period end matches it and this function would silently return None.


## 5. Fiscal period discovery (Decision B)

### 5.1 Why a rebalance date cannot supply a period end

Every query above needs a period end the caller supplies, but a factor at a rebalance
date holds 500 companies and none of their fiscal calendars: Apple's year ends in late
September, Costco's in late August, McKesson's in March, and each shifts by a few days a
year. `available_periods` and `latest_value_as_of` derive the period from the filings
themselves instead, promoted into `src/loaders/fundamentals.py` alongside everything else
in this section.

In [ ]:
# available_periods and latest_value_as_of live in src/loaders/fundamentals.py, validated
# against the fixes and tests below rather than defined here.
importlib.reload(F)
available_periods = F.available_periods
latest_value_as_of = F.latest_value_as_of


In [17]:
# One rebalance date, four filers, four different fiscal calendars. This is the
# problem the function exists to solve, stated as a table.
print("Annual net income available on 2015-03-31:\n")
for name, cik in [("Apple", 320193), ("Costco", 909832),
                  ("Coca-Cola", 21344), ("McKesson", 927653)]:
    facts = F.load_company_facts(cik)
    got = latest_value_as_of(facts, "net_income", "USD", "2015-03-31", "annual")
    print(f"  {name:10s} period ending {got[4]}  filed {got[1]}  {got[0]:>16,}")

# The look ahead boundary, which is the whole point. Apple's fiscal 2021 ended
# 2021-09-25 and its 10-K was filed 2021-10-29, so the answer must not move until
# the filing date itself.
aapl = F.load_company_facts(320193)
before = latest_value_as_of(aapl, "net_income", "USD", "2021-10-28", "annual")
on_day = latest_value_as_of(aapl, "net_income", "USD", "2021-10-29", "annual")
assert before[4] == "2020-09-26", "fiscal 2021 was not yet public on 2021-10-28"
assert on_day[4] == "2021-09-25", "fiscal 2021 was public on the day it was filed"
print(f"\nApple on 2021-10-28: period ending {before[4]}")
print(f"Apple on 2021-10-29: period ending {on_day[4]}   (the 10-K's filing date)")

# A company's available period steps forward when its report lands, not when its
# year closes. McKesson's fiscal 2015 ended 2015-03-31 and was filed in May.
mck = F.load_company_facts(927653)
steps = {d: latest_value_as_of(mck, "net_income", "USD", d, "annual")[4]
         for d in ["2015-03-31", "2015-04-30", "2015-05-31", "2015-06-30"]}
assert steps["2015-04-30"] == "2014-03-31" and steps["2015-05-31"] == "2015-03-31"
print("\nMcKesson, month by month:")
for d, end in steps.items():
    print(f"   rebalance {d} -> period ending {end}")

# offset reaches the prior period, which is what a growth factor needs.
current = latest_value_as_of(aapl, "net_income", "USD", "2022-01-01", "annual", offset=0)
prior = latest_value_as_of(aapl, "net_income", "USD", "2022-01-01", "annual", offset=1)
assert (current[4], prior[4]) == ("2021-09-25", "2020-09-26")
print(f"\nApple profit growth inputs as of 2022-01-01: "
      f"{prior[0]:,} ({prior[4]}) to {current[0]:,} ({current[4]})")

# A balance sheet concept has no annual or quarterly distinction, so the latest
# available period is the most recent quarterly balance sheet rather than the year
# end. That is the more current book value and the right one for book to price.
equity = latest_value_as_of(aapl, "stockholders_equity", "USD", "2015-03-31")
assert equity[4] == "2014-12-27", "expected the most recent quarterly balance sheet"
print(f"\nApple book value on 2015-03-31: {equity[0]:,} at {equity[4]}, "
      f"a quarter end rather than the fiscal year end")


Annual net income available on 2015-03-31:

  Apple      period ending 2014-09-27  filed 2015-01-28    39,510,000,000
  Costco     period ending 2014-08-31  filed 2014-10-15     2,058,000,000
  Coca-Cola  period ending 2014-12-31  filed 2015-02-25     7,098,000,000
  McKesson   period ending 2014-03-31  filed 2014-05-14     1,263,000,000

Apple on 2021-10-28: period ending 2020-09-26
Apple on 2021-10-29: period ending 2021-09-25   (the 10-K's filing date)

McKesson, month by month:
   rebalance 2015-03-31 -> period ending 2014-03-31
   rebalance 2015-04-30 -> period ending 2014-03-31
   rebalance 2015-05-31 -> period ending 2015-03-31
   rebalance 2015-06-30 -> period ending 2015-03-31

Apple profit growth inputs as of 2022-01-01: 57,411,000,000 (2020-09-26) to 94,680,000,000 (2021-09-25)

Apple book value on 2015-03-31: 123,328,000,000 at 2014-12-27, a quarter end rather than the fiscal year end


### 5.2 The derived route, and the adopted rule

`total_liabilities` and `gross_profit` are each resolvable two ways, a direct tag or a
figure derived from two other concepts. A period counts as available if either route
reaches it, or a filer relying entirely on the derived route would be undercounted here
even though `total_liabilities_as_of` and `gross_profit_as_of` already resolve it fine.

In [22]:
# Verifying the derived-fallback route in available_periods/latest_value_as_of above:
# total_liabilities_as_of and gross_profit_as_of already resolved these correctly, the
# gap was that period discovery did not know to look for periods reachable only
# through them.
dover = None
for path in sorted(F.FUNDAMENTALS_RAW_DIR.glob("*.json")):
    facts = json.loads(path.read_text())
    if facts.get("entityName") == "DOVER Corp":
        dover = facts
        break
assert dover is not None
last_filed = max(p["filed"] for tags in dover["facts"].values() for d in tags.values()
                  for pts in d["units"].values() for p in pts)
got = latest_value_as_of(dover, "total_liabilities", "USD", last_filed)
assert got == (5_990_212_000, "2026-07-23", "10-Q",
              "derived: total_assets - stockholders_equity", "2026-06-30")
print(f"Dover total_liabilities, period ending {got[4]}: {got[0]:,} via {got[3]}")

doordash = F.load_company_facts(1792789)
last_filed_dd = max(p["filed"] for tags in doordash["facts"].values() for d in tags.values()
                     for pts in d["units"].values() for p in pts)
got2 = latest_value_as_of(doordash, "gross_profit", "USD", last_filed_dd, period="quarterly")
assert got2 == (2_044_000_000, "2026-05-06", "10-Q",
               "derived: revenue - cost_of_revenue", "2026-03-31")
print(f"DoorDash gross_profit, period ending {got2[4]}: {got2[0]:,} via {got2[3]}")

Dover total_liabilities, period ending 2026-06-30: 5,990,212,000 via derived: total_assets - stockholders_equity
DoorDash gross_profit, period ending 2026-03-31: 2,044,000,000 via derived: revenue - cost_of_revenue


### 5.3 Measuring the rule's own output found four tag aliases

Running `available_periods` across the full cached universe, checking how many days
separate each company's last filing from the latest period each concept resolves for, is
itself a validation tool. It found gaps up to 6,330 days for concepts that should refresh
every quarter, not staleness in the ordinary sense but evidence the loader had stopped
seeing facts that exist. Tracing individual cases, not just the percentile table, found
four missing tag aliases, none of them related to period discovery itself.

In [18]:
# Decision B, staleness. A company that stops filing altogether leaves the index,
# so the universe layer protects against it. A company that keeps filing while
# ceasing to tag one concept is not protected: its latest available period stops
# advancing and every later rebalance receives the same old figure.
CONCEPT_PERIODS = (
    [(c, "annual") for c, k in F.CONCEPT_KIND.items() if k == "duration"]
    + [(c, "quarterly") for c, k in F.CONCEPT_KIND.items() if k == "duration"]
    + [(c, None) for c, k in F.CONCEPT_KIND.items() if k == "instant"]
)

rows = []
for path in sorted(F.FUNDAMENTALS_RAW_DIR.glob("*.json")):
    facts = json.loads(path.read_text())
    last_filed = max((p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts), default=None)
    if last_filed is None:
        continue
    for concept, period in CONCEPT_PERIODS:
        ends = available_periods(facts, concept, "USD", last_filed, period)
        gap = None if not ends else (dt.date.fromisoformat(last_filed)
                                     - dt.date.fromisoformat(ends[-1])).days
        rows.append({"entity": facts.get("entityName", "?"), "concept": concept,
                     "period": period or "instant", "last_filing": last_filed,
                     "latest_period": ends[-1] if ends else None, "gap_days": gap})

stale = pd.DataFrame(rows)
reported = stale[stale["gap_days"].notna()]
print(f"{len(stale)} company, concept and period combinations across "
      f"{stale['entity'].nunique()} companies")
print(f"{len(stale) - len(reported)} never reported at all\n")

# The distribution decides where a threshold would go, if one is warranted. As with
# the start date tolerance in Decision A, the useful outcome is a clear gap between
# a healthy cluster and a stale tail rather than a continuum.
print(reported.groupby("period")["gap_days"]
      .describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0).to_string())


12975 company, concept and period combinations across 864 companies
1706 never reported at all

            count   mean    std    min    50%    90%     95%     99%     max
period                                                                      
annual     3625.0  322.0  656.0   22.0  177.0  310.0  1223.0  3853.0  5690.0
instant    4026.0  141.0  555.0 -237.0   35.0   96.0   212.0  3637.0  5819.0
quarterly  3618.0  221.0  702.0   10.0   36.0  218.0  1308.0  4279.0  6247.0


In [19]:
# Where the instant tail actually lives: the specific companies and concepts,
# not just the percentile summary above. Instant concepts get a fresh fact
# nearly every quarter a company files, so a multi-year gap here needs an
# explanation the way Mastercard's frozen share count did in Decision C.
worst_instant = (reported[reported["period"] == "instant"]
                  .sort_values("gap_days", ascending=False)
                  .head(20))
print(worst_instant.to_string(index=False))

# Concentrated in a few concepts, or spread across the sample? Near-term debt
# maturity concepts are already known to be sparsely tagged (README, 60%
# coverage), a reasonable first guess for what is driving this, not an
# assumption.
print("\nGap days by concept (instant only):")
print(reported[reported["period"] == "instant"]
      .groupby("concept")["gap_days"]
      .describe(percentiles=[0.5, 0.9])[["count", "50%", "90%", "max"]]
      .round(0)
      .sort_values("max", ascending=False)
      .to_string())

# Concentrated in a few companies, or spread across the sample? A handful of
# companies dominating would echo Decision A's Costco/Apple shape; broad
# spread would look more like the structural multi-class shares gap instead.
threshold = reported["gap_days"].quantile(0.95)
print(f"\nCompanies in the top 5% of gap_days (>= {threshold:.0f} days):")
print(reported[reported["gap_days"] >= threshold]["entity"].value_counts().to_string())

# The known ifrs-full filer (Part 15): does it show up here as a large gap,
# having once filed under us-gaap before switching, rather than as "never
# reported at all"?
chilean_filer = F.load_company_facts(888746)["entityName"]
print(f"\nCIK 888746 ({chilean_filer}) in the staleness table:")
print(stale[stale["entity"] == chilean_filer].to_string(index=False))



                             entity                   concept  period last_filing latest_period  gap_days
           AMERIPRISE FINANCIAL INC long_term_debt_noncurrent instant  2026-06-05    2010-06-30    5819.0
          THE WESTERN UNION COMPANY    long_term_debt_current instant  2026-07-30    2010-12-31    5690.0
                 FIRST HORIZON CORP long_term_debt_noncurrent instant  2026-05-07    2010-12-31    5606.0
        RAYMOND JAMES FINANCIAL INC    long_term_debt_current instant  2026-05-14    2011-06-30    5432.0
                      CHUBB LIMITED    long_term_debt_current instant  2026-06-05    2011-09-30    5362.0
    VERTEX PHARMACEUTICALS INC / MA long_term_debt_noncurrent instant  2026-05-13    2011-09-30    5339.0
         DIGITAL REALTY TRUST, INC. long_term_debt_noncurrent instant  2026-07-01    2012-03-31    5205.0
           FRANKLIN RESOURCES, INC.    long_term_debt_current instant  2026-07-31    2012-06-30    5144.0
                      MetLife, Inc. long_term_

In [20]:
# Verifying the stockholders_equity fix (src/loaders/fundamentals.py): TAG_ALIASES is
# read at import time, so the running session needs a reload before it can see the new
# alias.
importlib.reload(F)

# Each of these topped the stockholders_equity tail in the staleness measurement above,
# all frozen at their last StockholdersEquity fact from 2008 to 2011. Queried at each
# company's own last filing date, so the answer is permanent: these are past filings
# already made, not a present-date snapshot.
for name, cik in [("CSX", 277948), ("Danaher", 313616),
                  ("Home Depot", 354950), ("Illinois Tool Works", 49826)]:
    facts = F.load_company_facts(cik)
    last_filed = max(p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts)
    ends = available_periods(facts, "stockholders_equity", "USD", last_filed)
    gap = (dt.date.fromisoformat(last_filed) - dt.date.fromisoformat(ends[-1])).days
    assert gap < 400, f"{name}'s stockholders_equity is still stale after the fix: {gap} days"
    print(f"  {name:20s} latest available period {ends[-1]}, {gap} days behind the last filing")

# The exact figure, resolved through the new tag rather than the old one.
csx = F.load_company_facts(277948)
got = F.concept_value_as_of(csx, "stockholders_equity", "USD", "2026-06-30", "2026-07-25")
assert got == (14_088_000_000, "2026-07-22", "10-Q",
              "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest")
print(f"\nCSX stockholders_equity, period ending 2026-06-30: {got[0]:,} via {got[3]}")

  CSX                  latest available period 2026-06-30, 22 days behind the last filing
  Danaher              latest available period 2026-06-26, 25 days behind the last filing
  Home Depot           latest available period 2026-05-03, 24 days behind the last filing
  Illinois Tool Works  latest available period 2026-03-31, 100 days behind the last filing

CSX stockholders_equity, period ending 2026-06-30: 14,088,000,000 via StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest


In [21]:
# Verifying the long term debt fix (src/loaders/fundamentals.py): ASC 842 moved most
# filers from "LongTermDebtCurrent"/"LongTermDebtNoncurrent" to the lease-inclusive
# "LongTermDebtAndCapitalLeaseObligations(Current)" tags around 2019, the same shape as
# the stockholders_equity fix above.
importlib.reload(F)

# Each of these topped the long_term_debt_current and long_term_debt_noncurrent tail in
# the staleness measurement, frozen at their last plain-tag fact from 2009 to 2019.
for name, cik in [("Home Depot", 354950), ("Lowe's", 60667), ("DTE Energy", 936340)]:
    facts = F.load_company_facts(cik)
    last_filed = max(p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts)
    for concept in ["long_term_debt_current", "long_term_debt_noncurrent"]:
        ends = available_periods(facts, concept, "USD", last_filed)
        gap = (dt.date.fromisoformat(last_filed) - dt.date.fromisoformat(ends[-1])).days
        assert gap < 400, f"{name}'s {concept} is still stale after the fix: {gap} days"
        print(f"  {name:12s} {concept:26s} latest period {ends[-1]}, {gap} days behind the last filing")

# The exact figures, resolved through the new lease-inclusive tag rather than the old one.
hd = F.load_company_facts(354950)
current = F.concept_value_as_of(hd, "long_term_debt_current", "USD", "2026-05-03", "2026-05-27")
noncurrent = F.concept_value_as_of(hd, "long_term_debt_noncurrent", "USD", "2026-05-03", "2026-05-27")
assert current == (5_178_000_000, "2026-05-27", "10-Q", "LongTermDebtAndCapitalLeaseObligationsCurrent")
assert noncurrent == (44_828_000_000, "2026-05-27", "10-Q", "LongTermDebtAndCapitalLeaseObligations")
print(f"\nHome Depot long term debt, period ending 2026-05-03: "
      f"{current[0]:,} current via {current[3]}, {noncurrent[0]:,} noncurrent via {noncurrent[3]}")

  Home Depot   long_term_debt_current     latest period 2026-05-03, 24 days behind the last filing
  Home Depot   long_term_debt_noncurrent  latest period 2026-05-03, 24 days behind the last filing
  Lowe's       long_term_debt_current     latest period 2026-05-01, 27 days behind the last filing
  Lowe's       long_term_debt_noncurrent  latest period 2026-05-01, 27 days behind the last filing
  DTE Energy   long_term_debt_current     latest period 2026-03-31, 71 days behind the last filing
  DTE Energy   long_term_debt_noncurrent  latest period 2026-03-31, 71 days behind the last filing

Home Depot long term debt, period ending 2026-05-03: 5,178,000,000 current via LongTermDebtAndCapitalLeaseObligationsCurrent, 44,828,000,000 noncurrent via LongTermDebtAndCapitalLeaseObligations


In [23]:
# Verifying the DebtCurrent fix (src/loaders/fundamentals.py): a large group of filers
# never adopted the ASC 842 lease-inclusive tag and instead moved the current portion of
# long term debt to the shorter "DebtCurrent".
importlib.reload(F)

for name, cik in [("Tyson Foods", 100493), ("Danaher", 313616)]:
    facts = F.load_company_facts(cik)
    last_filed = max(p["filed"] for tags in facts["facts"].values() for d in tags.values()
                      for pts in d["units"].values() for p in pts)
    ends = available_periods(facts, "long_term_debt_current", "USD", last_filed)
    gap = (dt.date.fromisoformat(last_filed) - dt.date.fromisoformat(ends[-1])).days
    assert gap < 400, f"{name}'s long_term_debt_current is still stale after the fix: {gap} days"
    print(f"  {name:12s} latest period {ends[-1]}, {gap} days behind the last filing")

# The exact figures, resolved through the new tag rather than the old one.
tyson = F.load_company_facts(100493)
got = F.concept_value_as_of(tyson, "long_term_debt_current", "USD", "2026-03-28", "2026-05-04")
assert got == (141_000_000, "2026-05-04", "10-Q", "DebtCurrent")
print(f"\nTyson Foods long_term_debt_current, period ending 2026-03-28: {got[0]:,} via {got[3]}")

# What remains unresolved past all three current-debt aliases is not a further missing
# alias: a genuine zero that a filer stopped re-tagging cannot be found by any tag name.
# Waters last tagged this concept at exactly 0 in 2021 and has not tagged it since.
waters = F.load_company_facts(1000697)
ends_w = available_periods(waters, "long_term_debt_current", "USD", "2026-01-01")
val_w = F.concept_value_as_of(waters, "long_term_debt_current", "USD", ends_w[-1], "2026-01-01")
assert ends_w[-1] == "2021-12-31" and val_w[0] == 0
print(f"Waters Corporation last tagged long_term_debt_current at {val_w[0]} on {ends_w[-1]}, "
      f"and has not re-tagged it since: not a defect, a lumpy figure genuinely at zero.")

  Tyson Foods  latest period 2026-03-28, 82 days behind the last filing
  Danaher      latest period 2026-06-26, 25 days behind the last filing

Tyson Foods long_term_debt_current, period ending 2026-03-28: 141,000,000 via DebtCurrent
Waters Corporation last tagged long_term_debt_current at 0 on 2021-12-31, and has not re-tagged it since: not a defect, a lumpy figure genuinely at zero.


In [24]:
# Verifying the SalesRevenueNet fix (src/loaders/fundamentals.py): revenue carried this
# name before roughly 2013, a tag missing from TAG_ALIASES until now, which made
# pre-2013 revenue unresolvable for the 370 cached companies that used it.
importlib.reload(F)

# Harley-Davidson tagged its 2009 second quarter revenue only under SalesRevenueNet;
# it never carries a "Revenues" fact that far back.
hd = F.load_company_facts(793952)
got = F.concept_value_as_of(hd, "revenue", "USD", "2009-06-28", "2009-10-01", "quarterly")
assert got == (1_153_645_000, "2009-07-31", "10-Q", "SalesRevenueNet")
print(f"Harley-Davidson revenue, quarter ending 2009-06-28: {got[0]:,} via {got[3]}")

# The same company's later revenue, well after its own switch to "Revenues", is
# unaffected: SalesRevenueNet is tried last and only reached when the other two aliases
# have nothing for the period queried.
got2 = F.concept_value_as_of(hd, "revenue", "USD", "2015-06-28", "2015-08-10", "quarterly")
assert got2 == (1_824_392_000, "2015-08-06", "10-Q", "Revenues")
print(f"Harley-Davidson revenue, quarter ending 2015-06-28: {got2[0]:,} via {got2[3]}")

Harley-Davidson revenue, quarter ending 2009-06-28: 1,153,645,000 via SalesRevenueNet
Harley-Davidson revenue, quarter ending 2015-06-28: 1,824,392,000 via Revenues


### 5.4 The usable column

A `fetched: True` CIK can still hold nothing usable: a foreign private issuer reporting
under `ifrs-full` instead of `us-gaap`, or, in one case, an unrelated entity reached
through a stale ticker-to-CIK mapping. `build_fundamentals`'s coverage report now
carries a `usable` column distinguishing this from a genuine fetch failure.

In [25]:
# Verifying the usable column added to build_fundamentals's coverage report
# (src/loaders/fundamentals.py): fetched no longer has to be conflated with usable.
importlib.reload(F)
coverage = F.build_fundamentals()
assert {"cik", "fetched", "usable"} <= set(coverage.columns)

# The three ifrs-full filers found this session, plus the empty-taxonomy duplicate XOM
# CIK, all fetched successfully but carry no us-gaap facts.
not_usable = {888746: "United Breweries", 1347557: "Pacific Airport Group",
              1816007: "Lufax Holding", 2115436: "XOM duplicate (ffd only)"}
for cik, label in not_usable.items():
    row = coverage[coverage["cik"] == cik]
    assert not row.empty and bool(row.iloc[0]["fetched"]) and not bool(row.iloc[0]["usable"]), label
    print(f"  cik={cik:8d} {label:28s} fetched=True  usable=False")

# A genuine us-gaap filer is usable, as expected.
aapl_row = coverage[coverage["cik"] == 320193]
assert bool(aapl_row.iloc[0]["fetched"]) and bool(aapl_row.iloc[0]["usable"])
print(f"  cik=320193   {'Apple':28s} fetched=True  usable=True")

print(f"\n{(coverage['fetched'] & ~coverage['usable']).sum()} of {coverage['fetched'].sum()} "
      f"fetched CIKs are not usable")

  cik=  888746 United Breweries             fetched=True  usable=False
  cik= 1347557 Pacific Airport Group        fetched=True  usable=False
  cik= 1816007 Lufax Holding                fetched=True  usable=False
  cik= 2115436 XOM duplicate (ffd only)     fetched=True  usable=False
  cik=320193   Apple                        fetched=True  usable=True

14 of 867 fetched CIKs are not usable


### 5.5 Promotion to `src`

`available_periods`, `latest_value_as_of`, and `DERIVED_FALLBACK` moved from this
notebook into `src/loaders/fundamentals.py`, and the four tag aliases found above went
into `TAG_ALIASES` alongside them. 10 tests were added to `tests/test_fundamentals.py`
for the period-discovery functions, taking the suite to 49. The cell that used to define
`available_periods` and `latest_value_as_of` locally now aliases the promoted versions,
so every verification cell above exercises the real, shipped code rather than a parallel
copy. Full evidence in Parts 18 through 20 of `notebooks/logs/fundamentals_construction.md`.

## 6. Summary